In [1]:
import os
from google.cloud import bigquery
from google.cloud import dataform_v1beta1 as dataform
import subprocess
import pyarrow.fs as fs


In [2]:
#q1
with open('/etc/os-release', 'r') as f:
    os_release = f.read()
os_release

'PRETTY_NAME="Ubuntu 24.04.3 LTS"\nNAME="Ubuntu"\nVERSION_ID="24.04"\nVERSION="24.04.3 LTS (Noble Numbat)"\nVERSION_CODENAME=noble\nID=ubuntu\nID_LIKE=debian\nHOME_URL="https://www.ubuntu.com/"\nSUPPORT_URL="https://help.ubuntu.com/"\nBUG_REPORT_URL="https://bugs.launchpad.net/ubuntu/"\nPRIVACY_POLICY_URL="https://www.ubuntu.com/legal/terms-and-policies/privacy-policy"\nUBUNTU_CODENAME=noble\nLOGO=ubuntu-logo\n'

In [3]:
PROJECT_ID = "sound-dialect-435221-g0"
REGION = "us-central1"
REPOSITORY_ID = "p8_longjam_agirish2"
WORKSPACE_ID = "p8_workid"
P8_BUCKET = os.environ["P8_BUCKET"]

In [4]:
#q2
result = subprocess.run(['lscpu'], capture_output=True, text=True)
result.stdout

'Architecture:                         x86_64\nCPU op-mode(s):                       32-bit, 64-bit\nAddress sizes:                        40 bits physical, 48 bits virtual\nByte Order:                           Little Endian\nCPU(s):                               4\nOn-line CPU(s) list:                  0-3\nVendor ID:                            GenuineIntel\nModel name:                           QEMU Virtual CPU version 2.5+\nCPU family:                           15\nModel:                                107\nThread(s) per core:                   1\nCore(s) per socket:                   4\nSocket(s):                            1\nStepping:                             1\nBogoMIPS:                             4400.00\nFlags:                                fpu de pse tsc msr pae mce cx8 apic sep mtrr pge mca cmov pat pse36 clflush mmx fxsr sse sse2 ht syscall nx lm constant_tsc nopl xtopology cpuid tsc_known_freq pni cx16 x2apic hypervisor lahf_lm cpuid_fault pti\nHypervisor vendor:    

In [5]:
#q3
# P8_BUCKET = "cs544-p8-agirish2-1765338582"
# P8_BUCKET = " p8-tianalongjam"

gcs = fs.GcsFileSystem()

with open('../wi-schools-raw.parquet', 'rb') as f:
    data = f.read()

with gcs.open_output_stream(f'{P8_BUCKET}/wi-schools-raw.parquet') as f:
    f.write(data)

file_info = gcs.get_file_info(fs.FileSelector(P8_BUCKET, recursive=False))
paths = [f.path for f in file_info]
paths

['p8-tianalongjam/wi-schools-raw.parquet']

In [6]:
#q4
file_info = gcs.get_file_info(f'{P8_BUCKET}/wi-schools-raw.parquet')
modification_time_ns = file_info.mtime_ns
modification_time_ns

1765571331721000000

In [7]:
dataset_id = f"{PROJECT_ID}.p8"
dataset = bigquery.Dataset(dataset_id)
dataset.location = "US"

try:
    dataset = client.create_dataset(dataset, exists_ok=True)
    print(f"✓ Dataset {dataset.dataset_id} is ready")
except Exception as e:
    print(f"Error: {e}")

Error: name 'client' is not defined


In [8]:
df_client = dataform.DataformClient()
workspace_path = f"projects/{PROJECT_ID}/locations/{REGION}/repositories/{REPOSITORY_ID}/workspaces/{WORKSPACE_ID}"
client = bigquery.Client(project=PROJECT_ID)

### Code for SQL Files that go to your Dataform

In [9]:
wi_counties_content = """config {
  type: "table",
  name: "wi_counties"
}

CREATE OR REPLACE TABLE `sound-dialect-435221-g0.p8.wi_counties` AS
SELECT *
FROM `bigquery-public-data.geo_us_boundaries.counties`
WHERE state_fips_code = '55'
"""

In [10]:
schools_content = f"""config {{
  type: "operations",
  name: "schools",
  hasOutput: true
}}

LOAD DATA OVERWRITE `{PROJECT_ID}.p8.schools` FROM FILES (
  format = 'PARQUET',
  uris = ['gs://{P8_BUCKET}/wi-schools-raw.parquet']
)
"""

In [11]:
wi_county_schools_content = """config {
  type: "table",
  name: "wi_county_schools"
}
CREATE OR REPLACE TABLE `sound-dialect-435221-g0.p8.wi_county_schools` AS
SELECT 
  s.*,
  ST_GEOGPOINT(s.longitude, s.latitude) AS location,
  c.county_name
FROM ${ref('schools')} s
JOIN ${ref('wi_counties')} c
ON ST_CONTAINS(c.county_geom, ST_GEOGPOINT(s.longitude, s.latitude))
"""

In [12]:
files = {
    "definitions/wi_counties.sqlx": wi_counties_content,
    "definitions/schools.sqlx": schools_content,
    "definitions/wi_county_schools.sqlx": wi_county_schools_content
}

In [13]:
for file_path, content in files.items():
    df_client.write_file(
        dataform.WriteFileRequest(
            workspace=workspace_path,
            path=file_path,
            contents=content.encode()
        )
    )
    print(f"Uploaded {file_path}")

Uploaded definitions/wi_counties.sqlx
Uploaded definitions/schools.sqlx
Uploaded definitions/wi_county_schools.sqlx


In [14]:
compilation = df_client.create_compilation_result(
    dataform.CreateCompilationResultRequest(
        parent=f"projects/{PROJECT_ID}/locations/{REGION}/repositories/{REPOSITORY_ID}",
        compilation_result=dataform.CompilationResult(
            workspace=workspace_path,
        ),
    )
)

assert compilation.compilation_errors == [], f"Compilation errors: {compilation.compilation_errors}"
print("Compilation successful!")

Compilation successful!


## Part 3

In [15]:
#q5
response = df_client.query_compilation_result_actions(
    request={"name": compilation.name}
)

dependencies = {}
for action in response.compilation_result_actions:
    action_name = action.target.name
    deps = [dep.name for dep in action.relation.dependency_targets]
    dependencies[action_name] = deps

dependencies

{'first_view': [],
 'schools': [],
 'second_view': ['first_view'],
 'wi_counties': [],
 'wi_county_schools': ['schools', 'wi_counties']}

In [16]:
# List all datasets in your project
datasets = list(client.list_datasets())

if datasets:
    print("Datasets in project:")
    for dataset in datasets:
        print(f"  - {dataset.dataset_id}")
else:
    print("No datasets found in project")

Datasets in project:
  - p8


In [17]:
tables = client.list_tables(f"{PROJECT_ID}.p8")

print("Tables in p8 dataset:")
for table in tables:
    print(f"  - {table.table_id}")

Tables in p8 dataset:
  - first_view
  - schools
  - second_view
  - wi_counties
  - wi_county_schools


In [18]:
# Check compilation
compilation = df_client.create_compilation_result(
    dataform.CreateCompilationResultRequest(
        parent=f"projects/{PROJECT_ID}/locations/{REGION}/repositories/{REPOSITORY_ID}",
        compilation_result=dataform.CompilationResult(
            workspace=workspace_path,
        ),
    )
)

print("Compilation errors:", compilation.compilation_errors)

# Check all actions
response = df_client.query_compilation_result_actions(
    request={"name": compilation.name}
)

print("\nActions found in compilation:")
for action in response.compilation_result_actions:
    print(f"  - {action.target.name}")
    if action.target.name == "wi_county_schools":
        print(f"    Dependencies: {[dep.name for dep in action.relation.dependency_targets]}")

Compilation errors: []

Actions found in compilation:
  - first_view
  - schools
  - second_view
  - wi_counties
  - wi_county_schools
    Dependencies: ['schools', 'wi_counties']


In [19]:


# Test the wi_counties query directly
query = """
CREATE OR REPLACE TABLE `sound-dialect-435221-g0.p8.wi_counties` AS
SELECT *
FROM `bigquery-public-data.geo_us_boundaries.counties`
WHERE state_fips_code = '55'
"""

try:
    query_job = client.query(query)
    query_job.result()  # Wait for completion
    print("✓ wi_counties table created successfully!")
except Exception as e:
    print(f"Error: {e}")

✓ wi_counties table created successfully!


In [20]:
tables = client.list_tables(f"{PROJECT_ID}.p8")

print("Tables in p8 dataset:")
for table in tables:
    print(f"  - {table.table_id}")

Tables in p8 dataset:
  - first_view
  - schools
  - second_view
  - wi_counties
  - wi_county_schools


In [21]:
#q6
query = """
SELECT COUNT(*) as county_count
FROM `sound-dialect-435221-g0.p8.wi_counties`
"""
result = client.query(query).to_dataframe()
int(result['county_count'].iloc[0])

72

In [22]:
# # Check the schema
# table = client.get_table(f"{PROJECT_ID}.p8.wi_county_schools")
# print("wi_county_schools columns:")
# for field in table.schema:
#     print(f"  - {field.name}: {field.field_type}")

In [23]:
#q7
query = f"""
SELECT COUNT(*) as school_count
FROM {PROJECT_ID}.p8.wi_county_schools
WHERE agency_type = 'Public school'
"""
result = client.query(query).to_dataframe()
int(result['school_count'].iloc[0])

2116

In [27]:
#q8
query = f"""
FROM `{PROJECT_ID}.p8.wi_county_schools`
|> WHERE agency_type = 'Public school' 
  AND school_type = 'High School'
|> AGGREGATE COUNT(*) as school_count GROUP BY county_name, city
|> WHERE school_count >= 3
|> AGGREGATE COUNT(*) as city_count GROUP BY county_name
|> WHERE city_count >= 2
|> SELECT county_name
|> ORDER BY county_name
"""
result = client.query(query).to_dataframe()
result['county_name'].tolist()

['Brown', 'Dane', 'Milwaukee', 'Waukesha']

In [25]:
#q9
job = client.query(query, job_config=job_config)
result = job.result()

bytes_per_query = job.total_bytes_processed
print(f"Bytes processed: {bytes_per_query}")

free_tier_bytes = 1024 ** 4
int(free_tier_bytes / bytes_per_query)

Bytes processed: 116370


9448411

In [26]:
#q10
query = f"""
SELECT 
  middle.school_name as middle_school,
  MIN_BY(high.school_name, ST_DISTANCE(middle.location, high.location)) as nearest_high_school
FROM `{PROJECT_ID}.p8.wi_county_schools` as middle
CROSS JOIN `{PROJECT_ID}.p8.wi_county_schools` as high
WHERE middle.county_name = 'Dane'
  AND middle.agency_type = 'Public school'
  AND middle.school_type = 'Middle School'
  AND high.county_name = 'Dane'
  AND high.agency_type = 'Public school'
  AND high.school_type = 'High School'
GROUP BY middle.school_name
ORDER BY middle.school_name
"""
result = client.query(query).to_dataframe()
dict(zip(result['middle_school'], result['nearest_high_school']))

{'Badger Ridge Middle': 'Verona Area High',
 'Badger Rock Middle': 'West High',
 'Belleville Middle': 'Belleville High',
 'Black Hawk Middle': 'Shabazz-City High',
 'Central Heights Middle': 'Prairie Phoenix Academy',
 'Cherokee Heights Middle': 'Capital High',
 'De Forest Middle': 'De Forest High',
 'Deerfield Middle': 'Deerfield High',
 'Ezekiel Gillespie Middle School': 'Vel Phillips Memorial High School',
 'Glacial Drumlin School': 'LaFollette High',
 'Glacier Creek Middle': 'Middleton High',
 'Hamilton Middle': 'Capital High',
 'Indian Mound Middle': 'McFarland High',
 'Innovative and Alternative Middle': 'Innovative High',
 'James Wright Middle': 'West High',
 'Kromrey Middle': 'Middleton High',
 'Marshall Middle': 'Marshall High',
 'Mount Horeb Middle': 'Mount Horeb High',
 'Nikolay Middle': 'Koshkonong Trails School',
 "O'Keeffe Middle": 'Innovative High',
 'Oregon Middle': 'Oregon High',
 'Patrick Marsh Middle': 'Prairie Phoenix Academy',
 'Prairie View Middle': 'Sun Prairie W